# InstantID Mobile — Export Pipeline

Notebook chạy toàn bộ pipeline convert InstantID → ONNX → INT8 → OnnxStream cho mobile.

**Runtime:** T4 GPU (Colab free) hoặc A100 (Colab Pro)

**Disk:** Pipeline tự dọn dẹp sau mỗi bước để tránh hết 112 GB.

## Cell 0 — Fix dependencies

In [ ]:
!pip install -q "diffusers>=0.28.0" "huggingface_hub>=0.23.0" "transformers>=4.40.0" onnxscript
import importlib, diffusers
importlib.reload(diffusers)
print(f"diffusers {diffusers.__version__}")

## Cell 1 — Clone repo + setup

In [ ]:
!git clone https://github.com/Hert4/Instantid-mobile
%cd Instantid-mobile
!./setup.sh --gpu

In [ ]:
# Kiểm tra disk
!df -h /content | tail -1
!echo "---"
!du -sh sdxl-base checkpoints sdxl-base-lcm 2>/dev/null || true

## Cell 2 — Fuse LCM-LoRA

In [ ]:
!python scripts/fuse_lcm_lora.py

# Verify fuse thành công
import os
assert os.path.exists("sdxl-base-lcm/model_index.json"), "Fuse failed — model_index.json not found!"
print("\n✅ Fuse thành công!")

In [ ]:
# Xóa SDXL gốc — đã fuse vào sdxl-base-lcm, không cần nữa (~6 GB)
!rm -rf ./sdxl-base
!echo "Freed ~6 GB" && df -h /content | tail -1

## Cell 3 — Export ONNX (từng phần để tránh OOM)

In [ ]:
# IP-Adapter + ControlNet không cần load full pipeline → export trước
!python scripts/export_all.py --only ip_adapter
!python scripts/export_all.py --only controlnet

In [ ]:
# Text encoders + VAE + UNet (cần load SDXL pipeline)
!python scripts/export_all.py --only text_encoders
!python scripts/export_all.py --only vae
!python scripts/export_all.py --only unet

In [ ]:
# Xóa sdxl-base-lcm + checkpoints — đã export hết sang ONNX (~8 GB)
!rm -rf ./sdxl-base-lcm
!echo "Freed sdxl-base-lcm" && df -h /content | tail -1

In [ ]:
# Verify ONNX files
!find onnx -name "*.onnx" -exec ls -lh {} \;

## Cell 4 — Quantize INT8

In [ ]:
!python scripts/quantize_all.py

In [ ]:
# Xóa ONNX FP16 gốc — đã quantize xong (~9 GB)
!rm -rf ./onnx
!echo "Freed ONNX FP16" && df -h /content | tail -1

## Cell 5 — Convert cho mobile (OnnxStream format)

In [ ]:
!python scripts/convert_for_onnxstream.py

In [ ]:
# Kiểm tra kết quả cuối cùng
!du -sh onnxstream-models/
!find onnxstream-models -name "*.yaml" -o -name "*.json" | head -20

## Cell 6 — Tải về máy

In [ ]:
import shutil
shutil.make_archive("onnxstream-models", "zip", "onnxstream-models")

from google.colab import files
files.download("onnxstream-models.zip")

### Hoặc lưu lên Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r onnxstream-models/ /content/drive/MyDrive/instantid-mobile/
print("✅ Saved to Google Drive!")